# Brain Stroke Prediction - Classification

## Section 1: Introduction and Dataset Overview

### Problem Statement

Stroke is a major cause of death and long-term disability worldwide. Early identification of high-risk individuals can enable timely intervention and improve patient outcomes.

This project develops a binary classification model to *predict* whether a patient is likely to suffer a stroke *(Class 1)* or not *(Class 0)* using demographic and clinical features.

The primary evaluation metric is **Recall for Class 1**, as correctly identifying stroke cases is more important than *avoiding false positives*. In a medical setting, missing a true stroke case (false negative) can have *serious consequences*.


### Dataset Description

The training dataset (`brain_stroke_data.csv`) contains 4,482 patient records with 11 columns. The unseen dataset (`brain_stroke_unseen.csv`) contains 499 records with 10 columns (the `stroke` target column is absent, as expected for held-out evaluation).

| Column | Type | Description |
|---|---|---|
| `gender` | Categorical | Patient sex: Male or Female |
| `age` | Numeric | Patient age in years |
| `hypertension` | Binary (0/1) | Whether the patient has hypertension |
| `heart_disease` | Binary (0/1) | Whether the patient has heart disease |
| `ever_married` | Categorical | Marital status: Yes or No |
| `work_type` | Categorical | Employment category: Private, Self-employed, Govt_job, children |
| `Residence_type` | Categorical | Urban or Rural residence |
| `avg_glucose_level` | Numeric | Average blood glucose level (mg/dL) |
| `bmi` | Numeric | Body Mass Index |
| `smoking_status` | Categorical | Smoking history: never smoked, formerly smoked, smokes, Unknown |
| `stroke` | Binary (0/1) | **Target variable** -> 1 = stroke occurred, 0 = no stroke |

The `smoking_status` value **"Unknown"** indicates that the patient's smoking history *was not recorded*; it is a legitimate data category and is **not treated as a missing value**. Similarly, `work_type = "children"` is a valid category representing minors and is retained as-is.

In [1]:
# Import all required libraries

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, precision_score,
                             recall_score, f1_score)

from imblearn.over_sampling import SMOTE

# Consistent plot style
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

In [3]:
# Load the training and unseen datasets using relative paths

df = pd.read_csv('brain_stroke_data.csv')
df_unseen = pd.read_csv('brain_stroke_unseen.csv')

print(f"Training dataset shape: {df.shape}")
print(f"Unseen dataset shape: {df_unseen.shape}")

Training dataset shape: (4482, 11)
Unseen dataset shape: (499, 10)


In [5]:
# training data inspection (few rows)
df.head()

,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,Female,46.0,0,0,Yes,Private,Urban,78.79,42.4,smokes,0
1,Female,65.0,0,0,Yes,Self-employed,Urban,248.24,27.0,smokes,0
2,Male,70.0,1,0,Yes,Self-employed,Rural,118.81,26.0,smokes,0
3,Male,47.0,0,0,Yes,Private,Urban,111.84,33.7,Unknown,0
4,Male,31.0,0,0,Yes,Govt_job,Urban,65.70,30.4,formerly smoked,0


In [6]:
# unseen data inspection (few rows)
df_unseen.head()

,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status
0,Female,49.0,0,0,Yes,Private,Rural,56.11,28.7,smokes
1,Male,75.0,0,0,Yes,Private,Rural,75.47,24.5,formerly smoked
2,Female,71.0,0,0,Yes,Private,Rural,100.61,19.2,Unknown
3,Female,12.0,0,0,No,children,Rural,85.97,35.7,Unknown
4,Male,63.0,0,0,Yes,Private,Rural,104.79,24.1,Unknown


In [7]:
# column data types
df.dtypes

gender                object
age                  float64
hypertension           int64
heart_disease          int64
ever_married          object
work_type             object
Residence_type        object
avg_glucose_level    float64
bmi                  float64
smoking_status        object
stroke                 int64
dtype: object

In [8]:
# statistics summary
df.describe()

,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke
count,4482.000000,4482.000000,4482.000000,4482.000000,4482.000000,4482.000000
mean,43.446693,0.098394,0.054663,105.761430,28.466064,0.049755
std,22.592327,0.297879,0.227347,44.943627,6.764658,0.217462
min,0.080000,0.000000,0.000000,55.120000,14.000000,0.000000
25%,26.000000,0.000000,0.000000,77.192500,23.600000,0.000000
50%,45.000000,0.000000,0.000000,91.680000,28.100000,0.000000
75%,61.000000,0.000000,0.000000,113.637500,32.575000,0.000000
max,82.000000,1.000000,1.000000,267.760000,48.900000,1.000000
